# Random Regret Minimisation (RRM) — Standalone Fit and Search

RRM models assume that decision-makers minimise **regret** rather than
maximise utility. For each alternative, the regret is the sum over all
attributes of the anticipated regret from choosing that alternative
compared to every other:

$$R_{ni} = \sum_j \sum_m \ln\left(1 + \exp\left[\beta_m(x_{njm} - x_{nim})\right]\right)$$

The choice probability is a softmax over negative regrets.

MLE is JAX-accelerated (vectorised pairwise regret computation).

In [ ]:
!pip install SearchLibrium --upgrade -q

In [ ]:
import numpy as np
import pandas as pd
from SearchLibrium import RandomRegret, Parameters, call_siman

## 1. Load data

RRM requires a long-format dataframe with one row per alternative.
The dataframe must contain an observation ID, an alternative ID,
a binary choice indicator, and the attribute columns.

In [ ]:
url = 'https://raw.githubusercontent.com/zahern/HypothesisX/refs/heads/main/data/TravelMode.csv'
df  = pd.read_csv(url)
df['AV']     = 1
df['CHOICE'] = df['choice'].map({'no': 0, 'yes': 1})
print(df.head(8))

## 2. Standalone RRM fit

In [ ]:
rrm = RandomRegret(df=df, short=False, normalize=True)
rrm.fit()
rrm.report()

## 3. Search over RRM specifications

The SA search explores which attribute subsets minimise BIC
under the RRM framework. All retained attributes are significant.

In [ ]:
varnames   = ['gcost', 'wait', 'vcost', 'travel']
choice_set = np.unique(df['mode']).tolist()

params = Parameters(
    criterions = [('bic', -1)],
    df         = df,
    varnames   = varnames,
    asvarnames = varnames,
    isvarnames = [],
    choice_set = choice_set,
    choices    = df['CHOICE'].values,
    alt_var    = df['mode'].values,
    choice_id  = df['individual'].values,
    base_alt   = None,
    models     = ['random_regret'],
    p_val      = 0.05,
    all_sig    = True,
)

best = call_siman(params, init_sol=None, id_num=1,
                  ctrl=(200, 0.001, 50, 10))

## 4. Compare MNL vs RRM

Run both model types in the same search and compare BIC scores.
A lower BIC for RRM indicates that regret-minimisation better
describes behaviour in this dataset.

In [ ]:
params_both = Parameters(
    criterions = [('bic', -1)],
    df         = df,
    varnames   = varnames,
    asvarnames = varnames,
    isvarnames = [],
    choice_set = choice_set,
    choices    = df['CHOICE'].values,
    alt_var    = df['mode'].values,
    choice_id  = df['individual'].values,
    base_alt   = None,
    models     = ['multinomial', 'random_regret'],  # search both
    p_val      = 0.05,
)

best_both = call_siman(params_both, init_sol=None, id_num=2,
                       ctrl=(300, 0.001, 80, 15))
print('Winner model:', best_both.get('model_n'))
print('BIC:         ', best_both.get('bic'))